In [7]:
# ============================================================
# SCRIPT 1: ALL CALLS - INTAKE LINES (FINAL)
# For Power BI Report with Buttons and Drill-Down
# ============================================================

import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

print("="*70)
print("SCRIPT 1: ALL CALLS - INTAKE LINES")
print("="*70)

# === LOAD ALL CALLS DATA ===
print("\n[1/7] LOADING ALL CALLS DATA...")

data_path = "/Users/bonsitukebeto/Library/CloudStorage/OneDrive-NorthwesternUniversity/Data/All Calls by Month"

folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

df_main = pd.DataFrame()

for i, f in enumerate(files, 1):
    print(f"   Reading file {i}: {f.stem}")
    try:
        if f.suffix.lower() == ".csv":
            df = pd.read_csv(f, header=0, dtype=str, engine="python")
        else:
            df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
        df_main = pd.concat([df_main, df], ignore_index=True)
    except Exception as e:
        print(f"   Error: {e}")

print(f"   ✓ Total rows loaded: {len(df_main):,}")

# === CONVERT TIMESTAMPS ===
print("\n[2/7] CONVERTING TIMESTAMPS...")

df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

print(f"   ✓ Date range: {df_main['Start time'].min()} to {df_main['Start time'].max()}")

# === CLASSIFY CALLS (BEFORE DURATION FILTERING) ===
print("\n[3/7] CLASSIFYING CALLS...")

df_main = df_main.sort_values(["Correlation ID", "Start time"])

def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    else:
        return "Internal"

df_main["TempCallType"] = df_main.apply(classify_call, axis=1)

earliest_calltype = df_main.drop_duplicates(subset="Correlation ID", keep="first")[["Correlation ID", "TempCallType"]].copy()

df_main = df_main.drop(columns=["TempCallType"])
df_main = df_main.merge(earliest_calltype.rename(columns={"TempCallType": "Call_Direction"}), on="Correlation ID", how="left")

print(f"   ✓ Call direction distribution:")
print(df_main['Call_Direction'].value_counts())

# === FILTER TO INBOUND CALLS ===
print("\n[4/7] FILTERING TO INBOUND CALLS...")

df_inbound = df_main[df_main["Call_Direction"] == "Inbound"].copy()

df_inbound["Duration"] = pd.to_numeric(df_inbound["Duration"], errors="coerce")
df_inbound = df_inbound[df_inbound["Duration"] > 0]

print(f"   ✓ Inbound rows (Duration > 0): {len(df_inbound):,}")
print(f"   ✓ Unique calls: {df_inbound['Correlation ID'].nunique():,}")

# === GET FIRST LEG PER CALL ===
print("\n[5/7] EXTRACTING FIRST LEG (INTAKE LINE DIALED)...")

df_first_leg = df_inbound.drop_duplicates(subset="Correlation ID", keep="first").copy()

print(f"   ✓ Unique calls: {len(df_first_leg):,}")

# === MAP PHONE NUMBERS ===
print("\n[6/7] MAPPING PHONE NUMBERS...")

number_map = {
    "13123478300": "Internal Voicemail",
    "13123411070": "Main Number",
    "13124235938": "Community Legal Clinics",
    "13122296300": "Direct Line to Front Desk",
    "1180": "Legal Menu (English)",
    "13125068646": "English Main Menu Transfer",
    "13124312299": "Farmworker/Migrant",
    "13122296079": "Nursing Home Ombudsman",
    "13122296344": "Bankruptcy Helpdesk",
    "13122296071": "Criminal Records",
    "13125068647": "Spanish Main Menu Transfer",
    "13123478340": "Veterans Project",
    "13122296014": "Markham Eviction Help Desk",
    "13123478309": "HIV Intake",
    "13122296072": "Juvenile Expungement (JEHD)",
    "13124235904": "Austin Intake",
    "2302": "Staff Directory (English)",
    "13123478347": "A2J Immigration",
    "18882652188": "A2J Immigration (Toll-Free)",
    "13124235900": "CLASP",
    "13123478392": "Education Law Referrals",
    "13124235909": "Fair Housing Intake",
    "13124312101": "OP Appeals Project",
    "13122296073": "Trafficking Survivors (TSAP)",
    "18004459025": "Farmworker/Migrant (Toll-Free)",
    "18884018200": "Nursing Home Ombudsman (Toll-Free)"
}

known_line_names = set(number_map.values())

def map_called_number(called_num):
    if pd.isna(called_num):
        return "Unknown"
    num_str = str(called_num).strip()
    if num_str in number_map:
        return number_map[num_str]
    else:
        return num_str

df_first_leg["Intake_Line"] = df_first_leg["Called number"].apply(map_called_number)
df_first_leg["Is_Unknown"] = ~df_first_leg["Intake_Line"].isin(known_line_names)

# Add Line Type for button filtering
df_first_leg["Line_Type"] = df_first_leg["Is_Unknown"].apply(lambda x: "Unknown Lines" if x else "Known Lines")

print(f"   ✓ Known lines: {(~df_first_leg['Is_Unknown']).sum():,}")
print(f"   ✓ Unknown lines: {df_first_leg['Is_Unknown'].sum():,}")

# === ADD TIME FEATURES ===
print("\n[7/7] CALCULATING RANKINGS AND MONTHLY AVERAGES...")

df_first_leg["Date"] = df_first_leg["Start time"].dt.date
df_first_leg["Year"] = df_first_leg["Start time"].dt.year
df_first_leg["Month"] = df_first_leg["Start time"].dt.month
df_first_leg["DayOfWeek"] = df_first_leg["Start time"].dt.weekday + 1
df_first_leg["Is_Weekday"] = df_first_leg["DayOfWeek"] <= 5

month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
               7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
df_first_leg["MonthName"] = df_first_leg["Month"].map(month_names)

# === CALCULATE RANKINGS ===

# Overall rankings (for ALL lines)
intake_totals = df_first_leg.groupby("Intake_Line").agg(
    Total_Calls=("Correlation ID", "nunique"),
    Is_Unknown=("Is_Unknown", "first"),
    Line_Type=("Line_Type", "first")
).reset_index()

intake_totals = intake_totals.sort_values("Total_Calls", ascending=False).reset_index(drop=True)
intake_totals["Intake_Rank"] = intake_totals.index + 1

def assign_rank_group(rank):
    if rank <= 5: return "Top 5"
    elif rank <= 10: return "Rank 6-10"
    elif rank <= 15: return "Rank 11-15"
    elif rank <= 20: return "Rank 16-20"
    else: return "Rank 21+"

intake_totals["Intake_Rank_Group"] = intake_totals["Intake_Rank"].apply(assign_rank_group)

# Known-specific rankings
known_totals = intake_totals[intake_totals["Is_Unknown"] == False].copy()
known_totals = known_totals.sort_values("Total_Calls", ascending=False).reset_index(drop=True)
known_totals["Known_Rank"] = known_totals.index + 1

def assign_known_rank(rank):
    if rank <= 5: return "Top 5 Known"
    elif rank <= 10: return "Rank 6-10 Known"
    elif rank <= 15: return "Rank 11-15 Known"
    elif rank <= 20: return "Rank 16-20 Known"
    else: return "Rank 21+ Known"

known_totals["Known_Rank_Group"] = known_totals["Known_Rank"].apply(assign_known_rank)

# Unknown-specific rankings
unknown_totals = intake_totals[intake_totals["Is_Unknown"] == True].copy()
unknown_totals = unknown_totals.sort_values("Total_Calls", ascending=False).reset_index(drop=True)
unknown_totals["Unknown_Rank"] = unknown_totals.index + 1

def assign_unknown_rank(rank):
    if rank <= 5: return "Top 5 Unknown"
    elif rank <= 10: return "Rank 6-10 Unknown"
    elif rank <= 20: return "Rank 11-20 Unknown"
    else: return "Rank 21+ Unknown"

unknown_totals["Unknown_Rank_Group"] = unknown_totals["Unknown_Rank"].apply(assign_unknown_rank)

# Merge all rankings back
intake_totals = intake_totals.merge(
    known_totals[["Intake_Line", "Known_Rank", "Known_Rank_Group"]],
    on="Intake_Line", how="left"
)
intake_totals = intake_totals.merge(
    unknown_totals[["Intake_Line", "Unknown_Rank", "Unknown_Rank_Group"]],
    on="Intake_Line", how="left"
)

# === CALCULATE MONTHLY AVERAGES (WEEKDAYS ONLY) ===
df_weekdays = df_first_leg[df_first_leg["Is_Weekday"]].copy()

weekdays_per_month = df_weekdays.groupby(["Year", "Month"]).agg(
    Weekdays_In_Month=("Date", "nunique")
).reset_index()

monthly = df_weekdays.groupby(["Year", "Month", "Intake_Line"]).agg(
    Monthly_Calls=("Correlation ID", "nunique")
).reset_index()

monthly = monthly.merge(weekdays_per_month, on=["Year", "Month"], how="left")
monthly["Monthly_Avg_Per_Weekday"] = (monthly["Monthly_Calls"] / monthly["Weekdays_In_Month"]).round(2)
monthly["MonthName"] = monthly["Month"].map(month_names)

# Create proper Date column for Power BI hierarchy (first day of month)
monthly["Date"] = pd.to_datetime(monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2) + "-01")

# Merge all rankings
monthly = monthly.merge(
    intake_totals[["Intake_Line", "Is_Unknown", "Line_Type", "Total_Calls",
                   "Intake_Rank", "Intake_Rank_Group", 
                   "Known_Rank", "Known_Rank_Group",
                   "Unknown_Rank", "Unknown_Rank_Group"]],
    on="Intake_Line", how="left"
)

# === EXPORT MAIN DATA ===
output_file = "PowerBI_Page1_IntakeLines.csv"
monthly.to_csv(output_file, index=False)

print(f"\n{'='*70}")
print(f"✅ EXPORTED: {output_file}")
print(f"{'='*70}")
print(f"   Total rows: {len(monthly):,}")
print(f"   Known lines: {monthly[monthly['Is_Unknown']==False]['Intake_Line'].nunique()}")
print(f"   Unknown lines: {monthly[monthly['Is_Unknown']==True]['Intake_Line'].nunique()}")

# === EXPORT SEPARATE DATE TABLE (for Power BI relationships) ===
date_table = monthly[["Date", "Year", "Month", "MonthName"]].drop_duplicates().sort_values("Date")
date_table["IsWeekday"] = True  # All our data is weekday-based
date_table.to_csv("PowerBI_DateTable.csv", index=False)

print(f"\n✅ EXPORTED: PowerBI_DateTable.csv")
print(f"   Date range: {date_table['Date'].min()} to {date_table['Date'].max()}")

print(f"\n   Columns in main export:")
for col in monthly.columns:
    print(f"      - {col}")

print(f"\n   Top 10 Intake Lines (Overall):")
for _, row in intake_totals.head(10).iterrows():
    line_type = "Known" if not row['Is_Unknown'] else "Unknown"
    print(f"      {row['Intake_Rank']:2}. [{line_type}] {row['Intake_Line']}: {row['Total_Calls']:,} calls")

SCRIPT 1: ALL CALLS - INTAKE LINES

[1/7] LOADING ALL CALLS DATA...
   Reading file 1: April 2024
   Reading file 2: April 2025
   Reading file 3: August 2024
   Reading file 4: August 2025
   Reading file 5: December 2024
   Reading file 6: February 2025
   Reading file 7: January 2025
   Reading file 8: July 2024
   Reading file 9: July 2025
   Reading file 10: June 2024
   Reading file 11: June 2025
   Reading file 12: March 2025
   Reading file 13: May 2024
   Reading file 14: May 2025
   Reading file 15: November 2024
   Reading file 16: October 2024
   Reading file 17: September 2024
   ✓ Total rows loaded: 1,001,537

[2/7] CONVERTING TIMESTAMPS...
   ✓ Date range: 2024-04-01 04:45:49.010000 to 2025-08-31 18:59:24.723000

[3/7] CLASSIFYING CALLS...
   ✓ Call direction distribution:
Call_Direction
Inbound     741568
Internal    136276
Outbound    123693
Name: count, dtype: int64

[4/7] FILTERING TO INBOUND CALLS...
   ✓ Inbound rows (Duration > 0): 716,266
   ✓ Unique calls: 299,6

In [12]:
import pandas as pd

df = pd.read_csv("PowerBI_Page1_IntakeLines_FINAL.csv")

# Check Main Number specifically
main_number = df[df["Intake_Line"] == "Main Number"]

print("=== Main Number Data ===")
print(main_number[["Year", "Month", "Monthly_Calls", "Weekdays_In_Month", "Monthly_Avg_Per_Weekday"]])

print(f"\n=== Summary ===")
print(f"Monthly_Calls range: {main_number['Monthly_Calls'].min()} - {main_number['Monthly_Calls'].max()}")
print(f"Monthly_Avg_Per_Weekday range: {main_number['Monthly_Avg_Per_Weekday'].min()} - {main_number['Monthly_Avg_Per_Weekday'].max()}")

=== Main Number Data ===
      Year  Month  Monthly_Calls  Weekdays_In_Month  Monthly_Avg_Per_Weekday
227   2024      4           5488                  9                   609.78
484   2024      5          11876                 23                   516.35
748   2024      6          11363                 20                   568.15
1008  2024      7          12561                 23                   546.13
1260  2024      8          12745                 22                   579.32
1510  2024      9          12394                 21                   590.19
1753  2024     10          12449                 23                   541.26
2000  2024     11           9694                 21                   461.62
2239  2024     12          10194                 22                   463.36
2493  2025      1          13097                 23                   569.43
2751  2025      2          10872                 20                   543.60
3009  2025      3          11644                 21

In [ ]:
# ============================================================
# SCRIPT 2: CAR - SERVICE CATEGORIES (FINAL)
# For Power BI Report with Buttons and Drill-Down
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

print("="*70)
print("SCRIPT 2: CAR - SERVICE CATEGORIES")
print("="*70)

# === LOAD CAR DATA ===
print("\n[1/6] LOADING CAR DATA...")

data_path = "/Users/bonsitukebeto/Library/CloudStorage/OneDrive-NorthwesternUniversity/Data/CAR_-_EP_Flow_Activity_Queue__Agent_Names"
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

df_car = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 
                                'Activity Start Timestamp', 'Queue Name', 'Agent Name', 'Termination Reason'])

for i, f in enumerate(files, 1):
    print(f"   Reading file {i}: {f.stem}")
    try:
        if f.suffix.lower() == ".csv":
            df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
        else:
            df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
        df_car = pd.concat([df_car, df], ignore_index=True)
    except Exception as e:
        print(f"   Error: {e}")

print(f"   ✓ Data files read: {i}")
print(f"   ✓ Total rows loaded: {len(df_car):,}")
print(f"   ✓ Unique sessions: {df_car['Contact Session ID'].nunique():,}")

# === CONVERT TIMESTAMPS ===
print("\n[2/6] CONVERTING TIMESTAMPS...")

df_car["Activity Start Timestamp"] = pd.to_datetime(
    df_car["Activity Start Timestamp"], 
    format="%Y/%m/%d %I:%M:%S %p", 
    errors="coerce"
)

# Sort by session and timestamp
df_car.sort_values(by=['Contact Session ID', 'Activity Start Timestamp'], inplace=True)
df_car.reset_index(inplace=True, drop=True)

# Add Date column
df_car['Date'] = df_car['Activity Start Timestamp'].dt.date

print(f"   ✓ Date range: {df_car['Activity Start Timestamp'].min()} to {df_car['Activity Start Timestamp'].max()}")

# === MAP EP NAMES TO CATEGORIES ===
print("\n[3/6] MAPPING EP NAMES TO CATEGORIES...")

# EP Name → Category mapping
ep_to_category = {
    'Legal Family Menu Telephony EP': 'Family',
    'Legal Housing Menu Telephony EP': 'Housing',
    'Legal Benefits Menu Telephony EP': 'Benefits',
    'Legal Employment Menu Telephony EP': 'Employment',
    'Legal Immigration Menu Telephony EP': 'Immigration',
    'Legal HIV Menu Telephony EP': 'HIV',
    'Other Legal Menu Telephony EP': 'Other Legal Issues',
}

# Activity Name → Subcategory mapping (for drill-down)
activity_to_subcategory = {
    # Family subcategories
    'FamilyMenu': 'Family - General',
    'DivorceOrParentingMenu': 'Divorce/Parenting',
    'ChildSupportMenu': 'Child Support',
    'SimpleDivorceMenu': 'Simple Divorce',
    'FamilySP': 'Family (Spanish)',
    
    # Housing subcategories
    'HousingMenu': 'Housing - General',
    'PreTenantMenu': 'Tenant Issues',
    'HousingSP': 'Housing (Spanish)',
    
    # Employment subcategories
    'EmploymentMenu': 'Employment - General',
    'WorkersCompMenu': 'Workers Compensation',
    'EmploymentSP': 'Employment (Spanish)',
    
    # Benefits subcategories
    'BenefitsMenu': 'Benefits - General',
    'BenefitsSP': 'Benefits (Spanish)',
    
    # Immigration subcategories
    'ImmigrationMenu': 'Immigration - General',
    'ImmigrationOtherMenu': 'Immigration - Other',
    'ImmigrationSP': 'Immigration (Spanish)',
    
    # HIV subcategories
    'HIVMenu': 'HIV - General',
    'HIVVoicemailTransfer': 'HIV Voicemail',
    
    # Other Legal Issues subcategories
    'TraffickingVoicemailTransfer': 'Trafficking',
    'TransferToSafeHaven': 'Safe Haven',
    'OtherLegalMenu': 'Other Legal - General',
}

# Service categories we care about
service_categories = list(ep_to_category.values())

# Map EP Name to Category
df_car['Category'] = df_car['EP Name'].map(ep_to_category)

# Map Activity Name to Subcategory
def get_subcategory(row):
    activity = row['Activity Name']
    category = row['Category']
    
    if pd.isna(activity) or activity == 'N/A':
        if pd.notna(category):
            return f"{category} - General"
        return None
    
    if activity in activity_to_subcategory:
        return activity_to_subcategory[activity]
    
    if pd.notna(category):
        return f"{category} - {activity}"
    
    return None

df_car['Subcategory'] = df_car.apply(get_subcategory, axis=1)

print(f"   ✓ EP Name distribution:")
print(df_car['EP Name'].value_counts().head(10))

# === GET DESTINATION CATEGORY PER SESSION ===
print("\n[4/6] IDENTIFYING DESTINATION CATEGORY PER CALL...")

# Get all unique session IDs
all_sessions = df_car['Contact Session ID'].unique()
print(f"   ✓ Total unique sessions: {len(all_sessions):,}")

# Filter to rows with service categories
df_service = df_car[df_car['Category'].isin(service_categories)].copy()

print(f"   ✓ Rows with service categories: {len(df_service):,}")
print(f"   ✓ Sessions reaching service categories: {df_service['Contact Session ID'].nunique():,}")

# Use drop_duplicates to get actual last row per session
df_destinations = df_service.drop_duplicates(subset='Contact Session ID', keep='last').copy()

print(f"   ✓ Sessions with service destinations: {len(df_destinations):,}")

# Capture sessions that never reached a service category
sessions_with_service = set(df_service['Contact Session ID'].unique())
sessions_without_service = set(all_sessions) - sessions_with_service

print(f"   ✓ Sessions WITHOUT service category (abandoned): {len(sessions_without_service):,}")

# Get the last row for sessions without service category
df_no_service = df_car[df_car['Contact Session ID'].isin(sessions_without_service)].copy()
df_abandoned = df_no_service.drop_duplicates(subset='Contact Session ID', keep='last').copy()

# Assign "Abandoned - No Selection" category
df_abandoned['Category'] = 'Abandoned - No Selection'
df_abandoned['Subcategory'] = 'Abandoned - No Selection'

print(f"   ✓ Abandoned sessions captured: {len(df_abandoned):,}")

# Combine destinations + abandoned
df_all_destinations = pd.concat([df_destinations, df_abandoned], ignore_index=True)

# Add Category Type for button filtering
df_all_destinations['Category_Type'] = df_all_destinations['Category'].apply(
    lambda x: 'Abandoned' if x == 'Abandoned - No Selection' else 'Service Category'
)

print(f"   ✓ Total sessions for analysis: {len(df_all_destinations):,}")

# === ADD TIME FEATURES ===
print("\n[5/6] ADDING TIME FEATURES...")

df_all_destinations["Year"] = df_all_destinations["Activity Start Timestamp"].dt.year
df_all_destinations["Month"] = df_all_destinations["Activity Start Timestamp"].dt.month
df_all_destinations["DayOfWeek"] = df_all_destinations["Activity Start Timestamp"].dt.weekday + 1
df_all_destinations["Is_Weekday"] = df_all_destinations["DayOfWeek"] <= 5

month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
               7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
df_all_destinations["MonthName"] = df_all_destinations["Month"].map(month_names)

# === CALCULATE MONTHLY AVERAGES ===
print("\n[6/6] CALCULATING MONTHLY AVERAGES AND RANKINGS...")

df_weekdays = df_all_destinations[df_all_destinations["Is_Weekday"]].copy()

# Count weekdays per month
weekdays_per_month = df_weekdays.groupby(["Year", "Month"]).agg(
    Weekdays_In_Month=("Date", "nunique")
).reset_index()

# --- CATEGORY LEVEL ---
monthly_category = df_weekdays.groupby(["Year", "Month", "Category"]).agg(
    Monthly_Calls=("Contact Session ID", "nunique")
).reset_index()

monthly_category = monthly_category.merge(weekdays_per_month, on=["Year", "Month"], how="left")
monthly_category["Monthly_Avg_Per_Weekday"] = (monthly_category["Monthly_Calls"] / monthly_category["Weekdays_In_Month"]).round(2)
monthly_category["MonthName"] = monthly_category["Month"].map(month_names)
monthly_category["Subcategory"] = monthly_category["Category"]  # Same as category for top level
monthly_category["Level"] = "Category"

# --- SUBCATEGORY LEVEL ---
monthly_subcategory = df_weekdays.groupby(["Year", "Month", "Category", "Subcategory"]).agg(
    Monthly_Calls=("Contact Session ID", "nunique")
).reset_index()

monthly_subcategory = monthly_subcategory.merge(weekdays_per_month, on=["Year", "Month"], how="left")
monthly_subcategory["Monthly_Avg_Per_Weekday"] = (monthly_subcategory["Monthly_Calls"] / monthly_subcategory["Weekdays_In_Month"]).round(2)
monthly_subcategory["MonthName"] = monthly_subcategory["Month"].map(month_names)
monthly_subcategory["Level"] = "Subcategory"

# Combine both levels
monthly_combined = pd.concat([monthly_category, monthly_subcategory], ignore_index=True)

# Create proper Date column for Power BI hierarchy
monthly_combined["Date"] = pd.to_datetime(
    monthly_combined["Year"].astype(str) + "-" + monthly_combined["Month"].astype(str).str.zfill(2) + "-01"
)

# --- CALCULATE CATEGORY RANKINGS ---
category_totals = df_weekdays.groupby("Category").agg(
    Total_Calls=("Contact Session ID", "nunique")
).reset_index().sort_values("Total_Calls", ascending=False)
category_totals["Category_Rank"] = range(1, len(category_totals) + 1)

def assign_category_rank_group(rank):
    if rank <= 3: return "Top 3 Categories"
    elif rank <= 5: return "Rank 4-5 Categories"
    else: return "Rank 6+ Categories"

category_totals["Category_Rank_Group"] = category_totals["Category_Rank"].apply(assign_category_rank_group)

# Add Category Type
category_totals["Category_Type"] = category_totals["Category"].apply(
    lambda x: 'Abandoned' if x == 'Abandoned - No Selection' else 'Service Category'
)

# --- CALCULATE SUBCATEGORY RANKINGS (within each category) ---
subcategory_totals = df_weekdays.groupby(["Category", "Subcategory"]).agg(
    Subcategory_Total_Calls=("Contact Session ID", "nunique")
).reset_index()

# Rank within each category
subcategory_totals["Subcategory_Rank"] = subcategory_totals.groupby("Category")["Subcategory_Total_Calls"].rank(
    ascending=False, method="dense"
).astype(int)

def assign_subcategory_rank_group(rank):
    if rank <= 3: return "Top 3 in Category"
    elif rank <= 5: return "Rank 4-5 in Category"
    else: return "Rank 6+ in Category"

subcategory_totals["Subcategory_Rank_Group"] = subcategory_totals["Subcategory_Rank"].apply(assign_subcategory_rank_group)

# Merge rankings into monthly data
monthly_combined = monthly_combined.merge(
    category_totals[["Category", "Category_Rank", "Category_Rank_Group", "Category_Type", "Total_Calls"]],
    on="Category", how="left"
)

monthly_combined = monthly_combined.merge(
    subcategory_totals[["Category", "Subcategory", "Subcategory_Rank", "Subcategory_Rank_Group", "Subcategory_Total_Calls"]],
    on=["Category", "Subcategory"], how="left"
)

# === EXPORT MAIN DATA ===
output_file = "PowerBI_Page2_ServiceCategories.csv"
monthly_combined.to_csv(output_file, index=False)

print(f"\n{'='*70}")
print(f"✅ EXPORTED: {output_file}")
print(f"{'='*70}")
print(f"   Total rows: {len(monthly_combined):,}")
print(f"   Categories: {monthly_combined['Category'].nunique()}")
print(f"   Subcategories: {monthly_combined['Subcategory'].nunique()}")

# === EXPORT SEPARATE DATE TABLE ===
date_table = monthly_combined[["Date", "Year", "Month", "MonthName"]].drop_duplicates().sort_values("Date")
date_table["IsWeekday"] = True
date_table.to_csv("PowerBI_Page2_DateTable.csv", index=False)

print(f"\n✅ EXPORTED: PowerBI_Page2_DateTable.csv")

# === EXPORT CATEGORY LOOKUP TABLE (for drill-down buttons) ===
category_lookup = category_totals[["Category", "Category_Rank", "Category_Rank_Group", "Category_Type", "Total_Calls"]].copy()
category_lookup.to_csv("PowerBI_Page2_CategoryLookup.csv", index=False)

print(f"\n✅ EXPORTED: PowerBI_Page2_CategoryLookup.csv")

print(f"\n   Columns in main export:")
for col in monthly_combined.columns:
    print(f"      - {col}")

print(f"\n   Category Rankings:")
for _, row in category_totals.iterrows():
    print(f"      {row['Category_Rank']}. {row['Category']}: {row['Total_Calls']:,} sessions")

print(f"\n   Level counts:")
print(monthly_combined['Level'].value_counts())

SCRIPT 2: CAR - SERVICE CATEGORIES

[1/6] LOADING CAR DATA...
   Reading file 1: CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25)
   Reading file 2: CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25)
   Reading file 3: CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25)
   Reading file 4: CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25)
   Reading file 5: CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25)
   Reading file 6: CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24)
   Reading file 7: CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24)
   Reading file 8: CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24)
   Reading file 9: CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24)
   Reading file 10: CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24)
   Reading file 11: CAR - EP, Flow, Activity, Queue, & Ag

In [14]:
import pandas as pd
import numpy as np

# Load the exported CSV
df = pd.read_csv("PowerBI_Page2_ServiceCategories.csv")

print("=== Before Fixes ===")
print(df.dtypes)
print(f"\nNull counts:\n{df.isnull().sum()}")

# Fix text columns - replace NaN with empty string
text_columns = ["Category", "Subcategory", "MonthName", "Level", 
                "Category_Rank_Group", "Category_Type", "Subcategory_Rank_Group"]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)
        df[col] = df[col].replace("nan", "")

# Keep numeric columns as numbers (NaN will be blank in Power BI)
numeric_columns = ["Year", "Month", "Monthly_Calls", "Weekdays_In_Month", 
                   "Monthly_Avg_Per_Weekday", "Category_Rank", "Total_Calls",
                   "Subcategory_Rank", "Subcategory_Total_Calls"]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Date column
df["Date"] = pd.to_datetime(df["Date"])

# Save fixed version
df.to_csv("PowerBI_Page2_ServiceCategories_FINAL.csv", index=False)

print("\n=== After Fixes ===")
print(df.dtypes)

print(f"\n✅ FINAL CSV saved: PowerBI_Page2_ServiceCategories_FINAL.csv")
print(f"\nTotal rows: {len(df):,}")
print(f"Categories: {df['Category'].nunique()}")
print(f"Subcategories: {df['Subcategory'].nunique()}")

print(f"\n=== Sample Data (Category Level) ===")
print(df[df["Level"] == "Category"][["Year", "Month", "Category", "Monthly_Avg_Per_Weekday", "Category_Rank_Group"]].head(10))

print(f"\n=== Sample Data (Subcategory Level) ===")
print(df[df["Level"] == "Subcategory"][["Year", "Month", "Category", "Subcategory", "Monthly_Avg_Per_Weekday"]].head(10))

print(f"\n=== Category Rankings ===")
cat_ranks = df[["Category", "Category_Rank", "Category_Rank_Group", "Total_Calls"]].drop_duplicates().sort_values("Category_Rank")
print(cat_ranks)

=== Before Fixes ===
Year                         int64
Month                        int64
Category                    object
Monthly_Calls                int64
Weekdays_In_Month            int64
Monthly_Avg_Per_Weekday    float64
MonthName                   object
Subcategory                 object
Level                       object
Date                        object
Category_Rank                int64
Category_Rank_Group         object
Category_Type               object
Total_Calls                  int64
Subcategory_Rank           float64
Subcategory_Rank_Group      object
Subcategory_Total_Calls    float64
dtype: object

Null counts:
Year                         0
Month                        0
Category                     0
Monthly_Calls                0
Weekdays_In_Month            0
Monthly_Avg_Per_Weekday      0
MonthName                    0
Subcategory                  0
Level                        0
Date                         0
Category_Rank                0
Category_Rank_G